# Baseline: Season Averages

This baseline approach uses each team's regular season average point differentials to predict the tournament margins.

$$\hat{m}_{A,B} = \overline{PD}_{A,season} - \overline{PD}_{B,season}$$

- where $\overline{PD}_{A,season}$ is team A's average point differentials in the regular season
- and $\overline{PD}_{B,season}$ is team B's average point differentials in the regular season

In [1]:
import pandas as pd

teams = pd.read_csv("Dataset/MTeams.csv")
regular = pd.read_csv("Dataset/MRegularSeasonCompactResults.csv")
ncaa_tourney = pd.read_csv("Dataset/MNCAATourneyCompactResults.csv")
tournament = pd.read_csv("Dataset/Tournament Game Point Differentials.csv")

print("regular\n", regular.head())
print("\n\n ncaa_tourney\n", ncaa_tourney.head())
print("\n\ntournament\n", tournament.head())

regular
    Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    1985      20     1228      81     1328      64    N      0
1    1985      25     1106      77     1354      70    H      0
2    1985      25     1112      63     1223      56    H      0
3    1985      25     1165      70     1432      54    H      0
4    1985      25     1192      86     1447      74    H      0


 ncaa_tourney
    Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    1985     136     1116      63     1234      54    N      0
1    1985     136     1120      59     1345      58    N      0
2    1985     136     1207      68     1250      43    N      0
3    1985     136     1229      58     1425      55    N      0
4    1985     136     1242      49     1325      38    N      0


tournament
    YEAR  CURRENT_ROUND  TEAM1_ID    TEAM1  TEAM1_SCORE  TEAM2_ID  \
0  2008              2        37  Memphis           68        43   
1  2008              4        13     UCLA           63  

In [2]:
id_to_name = {r.TeamID: r.TeamName for r in teams.itertuples(index=False)}

mm_teams = set() # only keep march madness teams
for row in ncaa_tourney.itertuples(index=False):
    mm_teams.add((row.Season, id_to_name.get(row.WTeamID)))
    mm_teams.add((row.Season, id_to_name.get(row.LTeamID)))

# each team's avg point differential from regular season games
regular_margins = []
for row in regular.itertuples(index=False):
    margin = row.WScore - row.LScore
    winner = id_to_name.get(row.WTeamID)
    loser = id_to_name.get(row.LTeamID)
    if (row.Season, winner) in mm_teams:
        regular_margins.append({"YEAR": row.Season, "TEAM": winner, "MARGIN": margin})
    if (row.Season, loser) in mm_teams:
        regular_margins.append({"YEAR": row.Season, "TEAM": loser, "MARGIN": -margin})
regular_margins = pd.DataFrame(regular_margins)
season_avg = regular_margins.groupby(["YEAR", "TEAM"], as_index=False)["MARGIN"].mean()
season_avg = {(r.YEAR, r.TEAM): r.MARGIN for r in season_avg.itertuples(index=False)}
season_avg_df = pd.DataFrame([{"YEAR": k[0], "TEAM": k[1], "MARGIN": v} for k, v in season_avg.items()])
print(season_avg_df.head())

   YEAR            TEAM    MARGIN
0  1985         Alabama  7.800000
1  1985         Arizona  7.185185
2  1985        Arkansas  3.636364
3  1985          Auburn  3.689655
4  1985  Boston College  5.269231


In [3]:
# predict tournament point differentials using season averages
pred_rows = []
for row in tournament.itertuples(index=False):
    avg1 = season_avg.get((row.YEAR, row.TEAM1))
    avg2 = season_avg.get((row.YEAR, row.TEAM2))
    if avg1 is None or avg2 is None:
        continue # skip if no regular season data

    pred_margin = avg1 - avg2
    pred_rows.append(
        {
            "YEAR": row.YEAR,
            "CURRENT_ROUND": row.CURRENT_ROUND,
            "TEAM1": row.TEAM1,
            "TEAM2": row.TEAM2,
            "TEAM1_AVG_PD": avg1,
            "TEAM2_AVG_PD": avg2,
            "PRED_MARGIN": pred_margin,
            "POINT_DIFFERENTIAL": row.POINT_DIFFERENTIAL,
            "ABS_ERROR": abs(row.POINT_DIFFERENTIAL - pred_margin),
        }
    )

preds = pd.DataFrame(pred_rows)
preds.head()

,YEAR,CURRENT_ROUND,TEAM1,TEAM2,TEAM1_AVG_PD,TEAM2_AVG_PD,PRED_MARGIN,POINT_DIFFERENTIAL,ABS_ERROR
0,2008,2,Memphis,Kansas,19.058824,19.515152,-0.456328,-7,6.543672
1,2008,4,UCLA,Memphis,14.393939,19.058824,-4.664884,-15,10.335116
2,2008,4,Kansas,North Carolina,19.515152,15.970588,3.544563,18,14.455437
3,2008,8,Xavier,UCLA,13.000000,14.393939,-1.393939,-19,17.606061
4,2008,8,Texas,Memphis,9.939394,19.058824,-9.119430,-18,8.880570


In [4]:
print("Overall margin MAE:", preds['ABS_ERROR'].mean())

print("Overall MAE by year:")
pd.DataFrame(preds.groupby('YEAR')['ABS_ERROR'].mean())

Overall margin MAE: 10.570436545405126
Overall MAE by year:


,ABS_ERROR
YEAR,
2008,10.187304
2009,11.602343
2010,9.208387
2011,11.250676
2012,8.338702
2013,11.897993
2014,10.856756
2015,8.357314
2016,11.058876


In [5]:
preds.to_csv("Predictions/baseline.csv", index=False)